# 2 · make — contact-prediction accuracy data

Aggregates [#245](https://github.com/Open-Athena/MarinFold/issues/245)'s published per-protein
R-precision into the table [`2_plot_rprecision.ipynb`](2_plot_rprecision.ipynb) draws. Nothing is
re-scored here — these are the numbers of record, read from the public bucket.

**Two protein classes, and why these ones.** *Natural* is the 314 natural FoldBench monomers
(`eval-val` + `eval-test`). *Designed* is FoldBench's 19 de novo monomers (`eval-denovo`) — the
only designed set where a baseline comparison is legitimate, because exp65's 396 designs are 20x
larger but 50.5 % of them were deposited on or before Protenix-v2's 2021-09-30 training cutoff,
which makes the baselines the contaminated party there.

`eval-test` is a held-out set with a read budget
([`eval_test_reads.md`](../../experiments/exp245_evals_foldbench_held_out_monomers/data/eval_test_reads.md)).
Re-displaying its published numbers is not a new read; scoring something new on it is.

CPU only.

In [ ]:
# Run from anywhere: figlib lives next to this notebook.
import sys
from pathlib import Path

HERE = Path.cwd() if (Path.cwd() / "figlib.py").exists() else Path("experiments/exp250_evals_exploration_notebook/figures")
sys.path.insert(0, str(HERE.resolve()))
import figlib

In [ ]:
# --- parameters -------------------------------------------------------------------------------
DATASET = "2_rprecision"
RANGE = "all"        # separation range: all | short | medium | long
METRIC = "R"         # R-precision; L, L/2, L/5 and AUC are also published
BOOTSTRAP_DRAWS = 2000
BOOTSTRAP_SEED = 0
CLASSES = {          # name -> the eval sets it pools
    "natural": ["eval-val", "eval-test"],
    "designed": ["eval-denovo"],
}

# #232's newer m2-p06 (step 363,000) beats the sweep final every published number here is for,
# but #232 deliberately left eval-test unscored, so `score_foldbench_rollouts.py` scored all 333
# monomers with it. Those rows join the published table under this name.
RESCORE = {"tag": "contacts-v1-exp232-m2-p06-train-1.5B",
           "predictor": "#232 m2-p06 step-363000 (decontaminated)"}
# The same pipeline rerun on the checkpoint #245 *did* publish, so the difference between this
# pipeline and theirs is measured rather than assumed. Nothing is drawn from it: it is the
# control that says whether the row above can sit beside published baselines at all.
VALIDATION = {"tag": "contacts-v1-exp232-m2-p06-1.5B-validate",
              "against": "#232 m2-p06 (decontaminated)"}

PARAMETERS = dict(range=RANGE, metric=METRIC, classes=CLASSES,
                  bootstrap_draws=BOOTSTRAP_DRAWS, bootstrap_seed=BOOTSTRAP_SEED,
                  rescore=RESCORE, validation=VALIDATION)
PARAMETERS

In [ ]:
import json

import pandas as pd

inputs = figlib.Inputs()
targets = figlib.load_foldbench_universe(inputs)
scores = figlib.load_foldbench_scores(inputs)
scores = scores[(scores.range == RANGE) & (scores.cut == METRIC)]
print(f"{len(targets)} units · {scores.predictor.nunique()} published predictors")
print(targets.eval_set.value_counts().to_string())

RESCORE_ROOT = (figlib.REPO / "experiments/exp250_evals_exploration_notebook"
                / "data/foldbench_rescore")


def load_rescored(tag, predictor):
    """One `score_foldbench_rollouts.py` run's per-protein rows, shaped like #245's table.

    Every shard file is recorded as an input, so a dataset built from a half-finished run shows
    up in the metadata rather than silently being short a few proteins.
    """
    directory = RESCORE_ROOT / tag
    shards = sorted(directory.glob("metrics-shard*.csv"))
    manifests = sorted(directory.glob("manifest-shard*.json"))
    if not shards:
        raise SystemExit(f"no metrics under {directory} — run `score_foldbench_rollouts.py "
                         f"--model ... --gpus N` on a GPU box first")
    frames = []
    for shard in shards:
        inputs.add_file(shard)
        frames.append(pd.read_csv(shard))
    for manifest in manifests:
        inputs.add_file(manifest)
    frame = pd.concat(frames, ignore_index=True)
    frame = frame[(frame.range == RANGE) & (frame.cut == METRIC)]
    recipe = json.loads(manifests[0].read_text())
    print(f"{tag}: {len(frame)} proteins over {len(shards)} shards, "
          f"{recipe['recipe']['n_rollouts']} rollouts each, {recipe['model']} on "
          f"{recipe['machine'].get('gpu', 'cpu')}")
    return frame.assign(predictor=predictor)[["dataset", "stem", "predictor", "range", "cut",
                                              "value"]]

In [ ]:
# --- does this pipeline agree with #245's? ------------------------------------------------------
# The rescored rows are drawn beside baselines nobody here re-ran, so the only honest way to use
# them is to first measure what the pipeline itself moves. This reruns the checkpoint #245
# published and compares protein by protein: a rollout evaluation is sampled, so the two will
# never be identical, and the question is whether the gap is small against the differences the
# figure claims.
validation = None
if VALIDATION and (RESCORE_ROOT / VALIDATION["tag"]).exists():
    mine = load_rescored(VALIDATION["tag"], VALIDATION["against"])
    theirs = scores[scores.predictor == VALIDATION["against"]]
    paired = mine.merge(theirs, on=["dataset", "stem"], suffixes=("_mine", "_published"))
    delta = paired.value_mine - paired.value_published
    validation = {"model": VALIDATION["against"], "n": int(len(paired)),
                  "mean_mine": float(paired.value_mine.mean()),
                  "mean_published": float(paired.value_published.mean()),
                  "mean_delta": float(delta.mean()),
                  "mean_absolute_delta": float(delta.abs().mean()),
                  "max_absolute_delta": float(delta.abs().max()),
                  "correlation": float(paired.value_mine.corr(paired.value_published))}
    print(f"pipeline check on {validation['n']} proteins: mean {validation['mean_mine']:.4f} "
          f"here vs {validation['mean_published']:.4f} published "
          f"(delta {validation['mean_delta']:+.4f}, mean |delta| "
          f"{validation['mean_absolute_delta']:.4f}, r={validation['correlation']:.3f})")
else:
    print("no validation run present — the rescored rows are unchecked against #245's pipeline")

scores = pd.concat([scores, load_rescored(RESCORE["tag"], RESCORE["predictor"])],
                   ignore_index=True)
print(f"{scores.predictor.nunique()} predictors after the rescore")

In [ ]:
rows, per_protein = [], []
for class_name, eval_sets in CLASSES.items():
    units = targets[targets.eval_set.isin(eval_sets)]
    keys = set(zip(units.dataset, units.stem))
    subset = scores[[key in keys for key in zip(scores.dataset, scores.stem)]]
    per_protein.append(subset.assign(protein_class=class_name))
    for predictor, group in subset.groupby("predictor"):
        mean, low, high = figlib.bootstrap_mean(group.value.values, BOOTSTRAP_DRAWS,
                                                BOOTSTRAP_SEED)
        rows.append(dict(protein_class=class_name, eval_sets="+".join(eval_sets),
                         predictor=predictor, n=len(group), value=mean,
                         ci_low=low, ci_high=high))

summary = pd.DataFrame(rows).sort_values(["protein_class", "value"], ascending=[True, False])
per_protein = pd.concat(per_protein, ignore_index=True)
print(summary.to_string(index=False, float_format=lambda v: f"{v:.3f}"))

In [ ]:
figlib.write_dataset(
    DATASET,
    notebook="2_make_rprecision_data.ipynb",
    parameters=PARAMETERS,
    inputs=inputs,
    files={
        "summary.csv": lambda path: summary.to_csv(path, index=False),
        "per_protein.csv": lambda path: per_protein.to_csv(path, index=False),
    },
    extra={
        "rescored": {"predictor": RESCORE["predictor"], "tag": RESCORE["tag"]},
        "pipeline_validation": validation,
        "metric": {"range": RANGE, "cut": METRIC,
                   "definition": "fraction of the N highest-confidence predicted contacts that "
                                 "are observed, where N is the protein's observed contact count"},
        "classes": {name: int((per_protein.protein_class == name).sum() //
                              max(1, per_protein[per_protein.protein_class == name]
                                  .predictor.nunique()))
                    for name in CLASSES},
    })